# Organising the Dataset

Before we can start downloading and preprocessing medical imaging data we need to first clean and process the dataset, reducing it to only sessions that meet our requirements. The requirements being:

1. Simplify clinical diagnoses into binary labels.
2. Find the most common MRI type and select sessions
3. For each MRI find a corresponding PET session that is within 6 months
4. Apply labeled diagnosis with historical diagnoses data

## Reading the data

The Oasis 3 dataset supplies a folder with data about the MRI and PET scans, as well as the subjects and diagnosis. This data is fragmented into multiple csv files and an MRI scan does not necessarily have a corresponding PET scan. Some entries contain errors or null values that need to be dropped or replaced.

In [ ]:
import os
import re
import pandas as pd
import numpy as np


dataset_path = "/home/kamran/projects/final-project/Datasets/Oasis-3"
data_path = "OASIS3_data_files/OASIS3_data_files/scans"
healthy = "OASIS_cohort_files/OASIS_cohort_files/scans/CogNorm-Cognitively_Normal_Cohorts/resources/csv/files/OASIS3_unchanged_CDR_cognitively_healthy.csv"
cdr = "UDSb4-Form_B4__Global_Staging__CDR__Standard_and_Supplemental/resources/csv/files/OASIS3_UDSb4_cdr.csv"
diag = "UDSd1-Form_D1__Clinician_Diagnosis___Cognitive_Status_and_Dementia/resources/csv/files/OASIS3_UDSd1_diagnoses.csv"
mri = "MRI-json-MRI_json_information/resources/csv/files/OASIS3_MR_json.csv"
pet = "PET-json-PET_json_information/resources/csv/files/OASIS3_PET_json.csv"
pup = "PUP-PUP_output/resources/csv/files/OASIS3_PUP.csv"

healthy_data = pd.read_csv(os.path.join(dataset_path, healthy))
cdr_data = pd.read_csv(os.path.join(dataset_path, data_path, cdr))
diag_data = pd.read_csv(os.path.join(dataset_path, data_path, diag))
mri_data = pd.read_csv(os.path.join(dataset_path, data_path, mri))
pet_data = pd.read_csv(os.path.join(dataset_path, data_path, pet))
pup_data = pd.read_csv(os.path.join(dataset_path, data_path, pup))

str_cols = ['OASISID', 'OASIS_session_label', 'dx1', 'dx2', 'dx3', 'dx4', 'dx5']
cdr_data[str_cols] = cdr_data[str_cols].apply(lambda s: s.astype(str))

# Find all cognitively healthy participants
healthy_ids = [str(x) for x in healthy_data['OASIS3_id'].unique()]
cdr_data_healthy = cdr_data[cdr_data['CDRTOT'] == 0]
cdr_healthy_ids = [str(x) for x in cdr_data_healthy.OASISID.unique()]

print("Received Healthy Diagnoses:",len(cdr_healthy_ids))
print("Unchanged Healthy:", len(healthy_ids))
difference = set(cdr_healthy_ids).difference(set(healthy_ids))
print("Converted:", len(difference))




Received Healthy Diagnoses: 1076
Unchanged Healthy: 755
Converted: 321


In [ ]:
# Find all records with dementia
cdr_data.replace({'.': 'A', np.nan: None}, inplace=True)

print("Initial Total: ", cdr_data.OASISID.count())
cdr_data = cdr_data[cdr_data['CDRTOT'] > 0]
print("Potential Dem. (CDR > 0):", cdr_data.OASISID.count())

def convert(d):
    """
    Classifies the diagnoses into 3 categories.
    
    1 = Dementia
    0.5 = Unclear
    0 = Non Dementia
    """
    keywords = ['AD dem', 'Vascular dem', 'Frontotemporal dem', 'Dementia']
    # reg_ex_dem = '^(AD dem|Vascular dem|Frontotemporal dem|(Active )?DLBD|Active PSNP|Dementia)'
    reg_ex_dem = f'^({"|".join(keywords)})'
    reg_ex_unc = '^uncertain*'
    if re.match(reg_ex_dem, str(d), re.I):
        return 1
    elif re.match(reg_ex_unc, str(d), re.I):
        return 0.5
    else:
        return 0

cdr_data['dementia'] = np.nan
unclear = []
for index, row in cdr_data.iterrows():
    assessment = convert(str(row['dx1']))

    agree = 0
    for i in range(2, 6):
        d = str(row[f'dx{i}'])
        if d == 'A' or d == None:
            agree += 1
        elif convert(str(row[f'dx{i}'])) >= assessment:
            agree += 1
    
    if agree > 1:
        cdr_data.at[index, 'dementia'] = assessment
    else:
        unclear.append(row['OASIS_session_label'])

print("Unclear:", len(unclear))
print(unclear[:5])
        



labels = set()
for i in range(1, 6):
    # print(cdr_data[f'dx{i}'].value_counts())
    labels = labels.union(cdr_data[f'dx{i}'].unique())

yes = set()
no = set()
uncertain = set()
for l in labels:
    if "unc" in str(l).lower():
        uncertain.add(str(l))
    elif re.match(reg_ex_dem, str(l), re.I):
        yes.add(str(l))
    else:
        no.add(str(l))

# for i in range(2,3):
#     print(cdr_data[f'dx{i}'].value_counts())

print(len(uncertain))
for l in uncertain:
    print(l)

# print("--------------")
# for l in labels:
#     if not "unc" in str(l).lower():
#         print(l)

for index, row in cdr_data.iterrows():
    pass

# print(healthy_ids)

Initial Total:  2146
Potential Dem. (CDR > 0): 2146
Unclear: 98
['OAS30013_UDSb4_d1384', 'OAS30035_UDSb4_d5796', 'OAS30035_UDSb4_d6244', 'OAS30038_UDSb4_d6357', 'OAS30038_UDSb4_d6678']
4
uncertain dementia
uncertain, possible NON AD dem
Unc: ques. Impairment
Unc: impair reversible


/tmp/ipykernel_163467/1114416355.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cdr_data.replace({'.': 'A', np.nan: None}, inplace=True)
